<a href="https://colab.research.google.com/github/Addychauhan/Addychauhan/blob/main/Customer_Churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing the Required Libarries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#Loading the Dataset

In [ ]:
# Load the raw Kaggle dataset
data = pd.read_csv("Churn_Modelling.csv")

#Basic Information

In [ ]:
# Top 5 Records
data.head(5)

In [ ]:
#Last 5 records
data.tail(5)

In [ ]:
#Shape of the Dataset
data.shape

In [ ]:
#Information of the Dataset
data.info()

In [ ]:
# Statistical Summary
data.describe()

In [ ]:
#Number of Null Values
data.isnull().sum()

In [ ]:
#Number of Duplicates
data.duplicated().sum()

#Data Cleaning/Preprocessing

In [ ]:
#DataType of the Columns
data.dtypes

In [ ]:
# Drop completely irrelevant columns that have zero predictive power
data = data.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

In [ ]:
data

#Exploratory Data Analysis

##Analysis of Numerical features

In [ ]:
data['Exited'].value_counts()

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Exited', data=data, width=0.4)
plt.title('Churn Distribution')
plt.show()

##Age Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['Age'], bins=30,kde=True)
plt.title('Distribution of Age')
plt.xlabel("Age")
plt.ylabel('Count')
plt.show()

##Balance Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['Balance'], bins=30, kde=True)
plt.title("Balance Distribution")
plt.xlabel("Balance")
plt.ylabel('Count')
plt.show()

##Credit Score Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['CreditScore'],
             bins=30,
             kde=True
             )
plt.title('Credit Score Distribution')
plt.xlabel("Credit Score")
plt.ylabel("Count")
plt.show()

#Analysis of Categorical Features

##Geography Distibution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Geography', data=data, width=0.4)
plt.show()

##Gender Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Gender', data=data, width=0.4)
plt.title("Gender Distribution")
plt.show()

##Bivariate Analysis

In [ ]:
fig, axarr=plt.subplots(3,2, figsize=(14,6))
sns.boxplot(x='Exited', y='CreditScore', data=data, ax=axarr[0][0])
sns.boxplot(x='Exited', y='Age', data=data, ax=axarr[0][1])

sns.boxplot(x='Exited', y='Tenure', data=data, ax=axarr[1][0])
sns.boxplot(x='Exited', y='Balance', data=data, ax=axarr[1][1])

sns.boxplot(x='Exited', y='NumOfProducts', data=data, ax=axarr[2][0])
sns.boxplot(x='Exited', y='EstimatedSalary', data=data, ax=axarr[2][1])
plt.tight_layout()
plt.show()

In [ ]:
data.columns

##Correlation Analysis

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(numeric_only=True),
          annot=True,
            cmap='coolwarm'
            )
plt.title("Correlation Heatmap")

In [ ]:
# Convert categorical text data (Geography, Gender) into numbers using One-Hot Encoding
df = pd.get_dummies(data, drop_first=True)
df

#Defining Feature Variables and Target Variable

In [ ]:
# Separate the dataset into Features (X) and Target (y)
X = df.drop(columns=['Exited'])
y = df['Exited']

In [ ]:
X

In [ ]:
y

#Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
from imblearn.combine import SMOTETomek

In [ ]:
# TRAIN-TEST SPLIT & SCALING

# Split into Train (80%) and Test (20%) sets before doing any scaling or resampling
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Scale the feature values to make sure distance-based metrics behave properly
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# APPLY SMOTETOMEK (BEST RESAMPLING)

# Apply SMOTETomek strictly to the training data to fix class imbalance and clear noise
smt = SMOTETomek(random_state=42)
X_train_smt, y_train_smt = smt.fit_resample(X_train_scaled, y_train)

print("Original Split:", pd.Series(y_train).value_counts().to_dict())
print("Cleaned Resampled Split:", pd.Series(y_train_smt).value_counts().to_dict())

In [ ]:
# TRAIN THE ADVANCED MODEL

# Initialize and fit the optimized Random Forest classifier
rf_model = RandomForestClassifier(n_estimators=300, max_depth=11, random_state=42, n_jobs=-1)
rf_model.fit(X_train_smt, y_train_smt)

In [ ]:
# AUTOMATED THRESHOLD TUNING

# Get the churn probability predictions for the original real test set
y_probabilities = rf_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Scan thresholds from 0.1 to 0.9 to locate the one that maximizes the F1-Score
thresholds = np.arange(0.1, 0.9, 0.01)
f1_scores = [f1_score(y_test, (y_probabilities >= t).astype(int)) for t in thresholds]

best_threshold = thresholds[np.argmax(f1_scores)]

In [ ]:
# Apply the discovered best threshold to make the final predictions
final_predictions = (y_probabilities >= best_threshold).astype(int)

In [ ]:
# PRINT EVALUATION METRICS

print(f"\n🏆 Optimal Decision Threshold Selected: {best_threshold:.3f}")
print("\n=== FINAL PRODUCTION CLASSIFICATION REPORT ===")
print(classification_report(y_test, final_predictions))
print(f"True Test Accuracy: {accuracy_score(y_test, final_predictions):.4f}")

In [ ]:
# PLOT FEATURE IMPORTANCE


df_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=df_importance, palette='viridis')
plt.title('🏆 Which Customer Features Drive Churn Decisions?')
plt.xlabel('Relative Importance Score')
plt.ylabel('Customer Feature')
plt.tight_layout()
plt.show()

In [ ]:
import pickle

# Bundle all model artifacts together
pipeline_artifacts = {
    'model': rf_model,
    'scaler': scaler,
    'best_threshold': best_threshold,
    'feature_names': list(X.columns)
}

# Write artifacts to a binary file
with open("churn_pipeline.pkl", "wb") as file:
    pickle.dump(pipeline_artifacts, file)

print("🏆 Production artifacts successfully saved to 'churn_pipeline.pkl'!")

In [ ]:
from google.colab import files
files.download('churn_pipeline.pkl')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import shutil
shutil.copy('churn_pipeline.pkl', '/content/drive/MyDrive/churn_pipeline.pkl')